# Práctica 3 · Etapa 5 — Métricas de regresión, curvas de evaluación y comparación

**3201 Modelos Inteligentes de Aprendizaje / 529 Redes Neuronales Modulares** · Maestría y Doctorado en Ciencias e Ingeniería (MyDCI) · Facultad de Ingeniería, Arquitectura y Diseño ·
Universidad Autónoma de Baja California · Ciclo 2026-2 · Versión 6

**Profesores:** Dr. Everardo Inzunza González · Dr. Enrique Efrén García Guerrero · Dr. Oscar Adrián Aguirre Castro

**Fase 5 de la sesión** · presupuesto: **25 min**

---

Aquí se abre, por primera y única vez, el bloque de prueba. Se calculan las métricas de regresión de `sklearn.metrics` —MSE, RMSE, MAE, R² y las «otras» que conviene conocer— más dos que exige el pronóstico (MASE y la habilidad frente a la climatología); se comparan los tres modelos con las tres líneas base, calculadas aquí mismo sobre la prueba; se diagnostica el sobreajuste y se generan las curvas de residuos, paridad, aprendizaje y validación.

| Necesita | Produce |
|---|---|
| artefactos de las Etapas 2, 3 y 4 | Tablas H e I · figuras F8 a F13 · `tablas/tabla_H_metricas_prueba.csv` · `artefactos/lineas_base_prueba_2038.csv` |

**Cómo leer este cuaderno.** Cada bloque abre con una celda de texto que explica *qué* se hace y *por qué*. Dentro del
código, el comentario a la derecha de cada instrucción dice para qué sirve esa línea. Las salidas marcadas con
**`← Anote`** son las que usted copia en la tabla indicada del manual; el resto sólo se ejecuta y se observa.
Ejecute las celdas **en orden**; si cambia algo, vuelva a ejecutar desde esa celda hacia abajo.

---
## Bloque 0 · Puesta en marcha

Esta celda se ejecuta **siempre primero**, en cualquier entorno. Detecta si está en Google Colab, en el JupyterHub del
laboratorio o en un Jupyter local; localiza la carpeta de la práctica (`P3_SeriesTiempo_v6`, la que descargó de
Classroom); crea `resultados/` con sus subcarpetas sin borrar nada; instala sólo lo que falte; fija la semilla e imprime
las versiones con que se ejecutó. Reproducir un resultado empieza por saber con qué se obtuvo.

> **En Colab**, cada cuaderno corre en su propia máquina virtual: los archivos que un cuaderno guarda sólo los ve el
> siguiente si viven en **Google Drive**. Por eso la carpeta de la práctica se sube a *Mi unidad* y esta celda la monta.

In [ ]:
# ── Bloque 0 · Puesta en marcha (Etapa 5) ─────────────────────────────────────────────
import sys, os, json, time, hashlib, warnings, subprocess, importlib  # utilidades estándar de Python
from pathlib import Path                                   # rutas que funcionan igual en Windows, Linux y Colab
T0 = time.perf_counter()                                   # cronómetro de este cuaderno
warnings.filterwarnings("ignore", category=FutureWarning)  # oculta sólo avisos de versiones futuras

try:
    import google.colab                                    # este módulo sólo existe dentro de Google Colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

CARPETA_PRACTICA = "P3_SeriesTiempo_v6"                    # la carpeta que descargó de Classroom
if EN_COLAB:
    from google.colab import drive                         # acceso a Google Drive desde Colab
    drive.mount("/content/drive")                          # la primera vez pide autorización
    RAIZ_DRIVE = Path("/content/drive/MyDrive")            # «Mi unidad»
    candidatas = [RAIZ_DRIVE / CARPETA_PRACTICA, RAIZ_DRIVE / "Colab Notebooks" / CARPETA_PRACTICA]
    candidatas += sorted(RAIZ_DRIVE.glob(f"*/{CARPETA_PRACTICA}"))  # también un nivel más abajo
else:
    aqui = Path.cwd().resolve()                            # carpeta desde donde se abrió el cuaderno
    candidatas = [aqui, *aqui.parents][:4]                 # el cuaderno vive en notebooks/: se busca hacia arriba
    candidatas += [p / CARPETA_PRACTICA for p in candidatas]
BASE = next((c for c in candidatas if (c / "datos").is_dir() and (c / "notebooks").is_dir()), None)
if BASE is None:
    raise FileNotFoundError("No encuentro la carpeta de la práctica (con datos/ y notebooks/). "
                            "Revise la Sección 4.3 del manual: en Colab súbala completa a «Mi unidad».")
DATOS, RES = BASE / "datos", BASE / "resultados"           # entradas y salidas de la práctica
FIG, TAB, ART = RES / "figuras", RES / "tablas", RES / "artefactos"  # figuras · tablas · archivos intermedios
for carpeta in (FIG, TAB, ART):
    carpeta.mkdir(parents=True, exist_ok=True)             # se crean si faltan; nunca se borra nada

# Bibliotecas: se instala sólo lo que falte (en Colab casi todo viene preinstalado).
NECESARIAS = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn", "xgboost": "xgboost",
               "matplotlib": "matplotlib", "seaborn": "seaborn", "statsmodels": "statsmodels",
               "openpyxl": "openpyxl", "scipy": "scipy"}
for modulo, paquete in NECESARIAS.items():
    try:
        importlib.import_module(modulo)                    # ¿ya está instalado?
    except ImportError:
        print("Instalando", paquete, "…")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

import numpy as np                                         # cálculo numérico con arreglos
import pandas as pd                                        # tablas de datos indexadas por fecha
import matplotlib                                          # motor de gráficas
import matplotlib.pyplot as plt                            # interfaz de gráficas
import sklearn, xgboost as xgb                             # scikit-learn y XGBoost
import seaborn, statsmodels, scipy                         # sólo para registrar sus versiones aquí

def _ver(v):
    return tuple(int("".join(ch for ch in p if ch.isdigit()) or 0) for p in v.split(".")[:2])
assert _ver(sklearn.__version__) >= (1, 4), "Se necesita scikit-learn ≥ 1.4 (root_mean_squared_error)."
assert _ver(xgb.__version__) >= (2, 0), "Se necesita XGBoost ≥ 2.0 (early_stopping_rounds en el constructor)."

SEMILLA = 42                                               # una sola semilla para toda la práctica
np.random.seed(SEMILLA)                                    # fija el azar de numpy
pd.set_option("display.width", 160)                        # tablas anchas sin cortes de renglón
pd.set_option("display.max_columns", 30)
COLORES = {"real": "#15241E", "XGBoost": "#17795C", "SVR": "#D9A21B", "Random Forest": "#9E2B25",
            "Ingenuo": "#9AA7A1", "Ingenuo estacional": "#5E6E67", "Climatología": "#165542"}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})  # estilo uniforme de las figuras

def guardar(fig, nombre):
    """Guarda la figura en resultados/figuras/ (PNG, 150 dpi) para insertarla en el manual."""
    ruta = FIG / f"{nombre}.png"
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    print("Figura guardada →", ruta.relative_to(BASE))

def _serializable(v):
    """Convierte tipos de numpy y pandas a tipos de Python; los no finitos (NaN, ±inf) quedan como None."""
    if isinstance(v, dict):
        return {str(k): _serializable(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)):
        return [_serializable(x) for x in v]
    if isinstance(v, (bool, np.bool_)):
        return bool(v)
    if isinstance(v, (int, np.integer)):
        return int(v)
    if isinstance(v, (float, np.floating)):
        return float(v) if np.isfinite(v) else None
    if isinstance(v, pd.Timestamp):
        return v.isoformat()
    return v

def registrar(clave, valor):
    """Añade un resultado a resultados/registro_P3.json (se acumula entre cuadernos; JSON estricto, sin NaN)."""
    ruta = RES / "registro_P3.json"
    reg = json.loads(ruta.read_text(encoding="utf-8")) if ruta.exists() else {}
    reg[clave] = _serializable(valor)
    ruta.write_text(json.dumps(reg, ensure_ascii=False, indent=1, allow_nan=False), encoding="utf-8")

def md5(ruta):
    return hashlib.md5(Path(ruta).read_bytes()).hexdigest()  # huella del archivo

if EN_COLAB:
    ENTORNO = "Google Colab"
elif "JUPYTERHUB_USER" in os.environ:
    ENTORNO = "JupyterHub"                                 # sesión abierta en el JupyterHub del laboratorio
elif "ai_env" in sys.executable:
    ENTORNO = "Kernel ai_env del laboratorio"              # el mismo kernel, fuera de una sesión del Hub
else:
    ENTORNO = "Jupyter local"
VERSIONES = {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
              "scikit-learn": sklearn.__version__, "xgboost": xgb.__version__, "matplotlib": matplotlib.__version__,
              "seaborn": seaborn.__version__, "statsmodels": statsmodels.__version__, "scipy": scipy.__version__}
registrar("etapa5_entorno", {"entorno": ENTORNO, **VERSIONES})
print("Entorno        :", ENTORNO)                         # ← Anote (Tabla A)
print("Carpeta base   :", BASE)
for k, v in VERSIONES.items():
    print(f"{k:<15}:", v)                                  # ← Anote (Tabla A): Python, scikit-learn, XGBoost y pandas

> **Regla de decisión, fijada antes de abrir la prueba.** El modelo que «se lleva» es el que tuvo menor RMSE en
> **validación** (Tabla G, Etapa 4); la métrica principal es el **RMSE** y la secundaria el **MAE**. La Tabla H que
> sigue es **descriptiva**: dice cómo le fue a ese modelo —y a los demás— en datos que nadie miró al decidir. Si otro
> modelo resulta mejor en la prueba, eso se reporta como observación, no se usa para cambiar de modelo: cambiarlo
> ahora convertiría la prueba en una segunda validación.

In [ ]:
from sklearn.model_selection import train_test_split
necesarios = ["caracteristicas_2038.csv", "serie_2038_limpia.csv", "particion_2038.json", "predicciones_2038.csv",
              "predicciones_trval_2038.csv", "lineas_base_val_2038.csv", "hiperparametros_2038.json"]
faltan = [n for n in necesarios if not (ART / n).exists()]
if faltan:
    raise FileNotFoundError(f"Faltan {faltan}: ejecute primero las Etapas 2, 3 y 4.")
tabla = pd.read_csv(ART / "caracteristicas_2038.csv", index_col="FECHA", parse_dates=True)
y = tabla.pop("y"); X = tabla
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, shuffle=False)  # la misma partición
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, shuffle=False)
guardada = json.loads((ART / "particion_2038.json").read_text(encoding="utf-8"))
for nombre, parte in {"entrenamiento": y_train, "validación": y_val, "prueba": y_test}.items():
    assert [f"{t:%Y-%m}" for t in parte.index] == guardada[nombre]["fechas"], f"Bloque «{nombre}» distinto al de la Etapa 3"
X_trval, y_trval = pd.concat([X_train, X_val]), pd.concat([y_train, y_val])  # desarrollo (ajuste del modelo final)
serie_regular = pd.read_csv(ART / "serie_2038_limpia.csv", index_col="FECHA", parse_dates=True)  # rejilla mensual + bandera
pred = pd.read_csv(ART / "predicciones_2038.csv", index_col="FECHA", parse_dates=True)  # pronósticos de prueba (Etapa 4)
pred_trval = pd.read_csv(ART / "predicciones_trval_2038.csv", index_col="FECHA", parse_dates=True)
HP = json.loads((ART / "hiperparametros_2038.json").read_text(encoding="utf-8"))
assert (pred.index == y_test.index).all() and np.allclose(pred["real"], y_test)  # mismas fechas y valores
NOMBRES = ["XGBoost", "SVR", "Random Forest"]
BASES = ["Ingenuo", "Ingenuo estacional", "Climatología"]
ELEGIDO = HP["elegido_en_validacion"]  # el modelo que la regla de decisión lleva a la prueba
print("Meses de prueba:", len(y_test), "·", f"{y_test.index[0]:%Y-%m} a {y_test.index[-1]:%Y-%m}",
      "· modelo elegido en validación:", ELEGIDO)
huecos = pd.date_range(y_test.index[0], y_test.index[-1], freq="MS").difference(y_test.index)  # meses sin evaluar
print("Meses de calendario sin evaluar dentro de la prueba (objetivo imputado):", [f"{t:%Y-%m}" for t in huecos])

---
## 5.0 Las líneas base en la prueba

Las tres reglas de la Etapa 3 se calculan ahora sobre el bloque de prueba, con la **misma información** que tuvieron
los modelos al reentrenarse: la climatología usa las medias mensuales de entrenamiento + validación; los ingenuos
toman `lag_1` y `lag_12` de la propia tabla, que ya están alineados por fecha.

In [ ]:
def climatologia(y_ajuste, indice):
    """Media de cada mes calendario en y_ajuste, asignada a las fechas de `indice`."""
    medias = y_ajuste.groupby(y_ajuste.index.month).mean()  # 12 valores, uno por mes
    return pd.Series(medias.reindex(indice.month).values, index=indice)

pred["Ingenuo"] = X_test["lag_1"]                           # y(t−1)
pred["Ingenuo estacional"] = X_test["lag_12"]               # y(t−12)
pred["Climatología"] = climatologia(y_trval, X_test.index)  # medias de entrenamiento + validación
pred[BASES].to_csv(ART / "lineas_base_prueba_2038.csv", index_label="FECHA")  # las usa la Etapa 6 (P2)
pred[["real"] + BASES].head()

---
## 5.1 Las métricas de regresión, una por una

Sea $e_t = y_t - \hat y_t$ el error del mes $t$ y $n$ el número de meses evaluados.

| Métrica | Fórmula | Unidades | Cómo se lee |
|---|---|---|---|
| **MSE** | $\frac{1}{n}\sum e_t^2$ | mm² | eleva al cuadrado: los errores grandes pesan mucho más |
| **RMSE** | $\sqrt{\mathrm{MSE}}$ | mm | el MSE en las unidades de la variable |
| **MAE** | $\frac{1}{n}\sum \lvert e_t\rvert$ | mm | error típico; menos sensible a los extremos |
| **R²** | $1-\frac{\sum e_t^2}{\sum (y_t-\bar y)^2}$ | — | **reducción relativa del error cuadrático frente a predecir la media del bloque evaluado**; puede ser negativo |
| **RSS** | $\sum e_t^2$ | mm² | suma de residuos al cuadrado ($n \cdot$ MSE) |
| MedAE | mediana de $\lvert e_t\rvert$ | mm | error «del mes típico», casi inmune a los extremos |
| Error máximo | $\max \lvert e_t\rvert$ | mm | el peor mes |
| Varianza explicada | $1-\frac{\mathrm{Var}(e)}{\mathrm{Var}(y)}$ | — | como el R² pero ignora el sesgo medio |
| MAPE | $\frac{100}{n}\sum \lvert e_t/y_t\rvert$ | % | **no está definido si algún $y_t = 0$** |
| sMAPE | $\frac{100}{n}\sum \frac{2\lvert e_t\rvert}{\lvert y_t\rvert+\lvert\hat y_t\rvert}$ | % | versión simétrica; acotada entre 0 y 200 |
| **MASE** | $\mathrm{MAE} / \mathrm{MAE}_{\text{ingenuo estacional, ajuste}}$ | — | < 1: mejor que repetir el mismo mes del año anterior (escala medida dentro de muestra) |
| **Habilidad vs. climatología** | $1-\mathrm{MSE}/\mathrm{MSE}_{\text{clim}}$ | — | > 0: mejor que pronosticar «un mes típico» |

Las primeras cinco son las que pide la práctica; las demás amplían la lectura. El R² se lee en toda la práctica como
reducción relativa del error frente a la media del bloque evaluado, no como «varianza explicada» (Etapa 1, sección 1.8).

**La escala del MASE se calcula por calendario.** Es el MAE del ingenuo estacional sobre los datos de ajuste: la media
de $\lvert y_t - y_{t-12}\rvert$ para los pares de meses que distan exactamente doce meses **en el calendario**, no doce
posiciones en una tabla que puede tener huecos. Se usa la rejilla mensual regular guardada en la Etapa 2
(`serie_2038_limpia.csv`), se exige que los dos meses del par sean **observados** (no imputados) y que $t$ no pase del
fin de la validación. La función devuelve también cuántos pares entraron.

In [ ]:
from sklearn.metrics import (mean_squared_error, root_mean_squared_error, mean_absolute_error, r2_score,
                             median_absolute_error, max_error, explained_variance_score,
                             mean_absolute_percentage_error)

def escala_mase(serie_regular, hasta, m=12):
    """MAE del ingenuo estacional por calendario: pares (t, t−m) observados, con t ≤ hasta. Devuelve (escala, n pares)."""
    s = serie_regular["PRECIP_mm"]                       # rejilla mensual regular (una fila por mes)
    observado = ~serie_regular["imputado"].astype(bool)  # True donde el dato es real
    diferencia = (s - s.shift(m)).loc[:hasta]            # y(t) − y(t−m), alineado por calendario
    validos = (observado & observado.shift(m, fill_value=False)).loc[:hasta]  # ambos meses observados
    return float(diferencia[validos].abs().mean()), int(validos.sum())

def smape(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    den = np.abs(y) + np.abs(p)                          # si real y pronóstico son 0, el término vale 0
    return 100 * np.mean(np.where(den == 0, 0.0, 2 * np.abs(y - p) / np.where(den == 0, 1, den)))

def metricas(y, p, escala, mse_clim=None):
    """Todas las métricas de la Tabla H para un pronóstico p de la serie y, con la escala del MASE ya calculada."""
    y, p = np.asarray(y, float), np.asarray(p, float)
    r = {"n": int(len(y)),                               # meses evaluados
         "MSE": mean_squared_error(y, p),                # promedio de e²
         "RMSE": root_mean_squared_error(y, p),          # raíz del MSE, en mm
         "MAE": mean_absolute_error(y, p),               # promedio de |e|
         "R2": r2_score(y, p),                           # 1 − RSS/TSS frente a la media del bloque evaluado
         "RSS": float(np.sum((y - p) ** 2)),             # suma de e²
         "MedAE": median_absolute_error(y, p),           # mediana de |e|
         "MaxErr": max_error(y, p),                      # el peor mes
         "VarExp": explained_variance_score(y, p),       # como el R², sin el sesgo medio
         "sMAPE_%": smape(y, p),                         # porcentaje simétrico, acotado
         "MASE": mean_absolute_error(y, p) / escala}     # MAE relativo al ingenuo estacional de ajuste
    if mse_clim is not None:
        r["Hab_clim"] = 1 - r["MSE"] / mse_clim          # habilidad frente a la climatología
    return r

ESCALA, N_PARES = escala_mase(serie_regular, hasta=y_val.index[-1])  # con los datos de desarrollo
print(f"Escala del MASE (MAE del ingenuo estacional hasta {y_val.index[-1]:%Y-%m}): {ESCALA:.3f} mm · {N_PARES} pares")  # ← Anote (Tabla H: escala; pares y fecha límite al pie)

In [ ]:
# MAPE con ceros: scikit-learn no falla, pero divide entre un número diminuto (épsilon de máquina).
ceros = int((y_test == 0).sum())
print(f"Meses de prueba con precipitación 0: {ceros} de {len(y_test)}")  # ← Anote (Tabla H)
mape_rf = mean_absolute_percentage_error(y_test, pred["Random Forest"])  # proporción (×100 sería el porcentaje)
print(f"MAPE del Random Forest según scikit-learn: {mape_rf:.3e} (proporción)")  # ← Anote (Tabla H): ¿es interpretable?

> **Atención: el MAPE no se reporta en esta serie.** Con meses de lluvia cero, cada uno de esos meses aporta un
> cociente $|e_t|/\varepsilon$ gigantesco y el promedio deja de significar algo. La función devuelve un número, no un
> error: el que debe detectarlo es usted. En el consumo eléctrico de la Etapa 1, que nunca vale cero, el MAPE sí era útil.

---
## 5.2 Tabla comparativa en el bloque de prueba (Tabla H)

In [ ]:
mse_clim = mean_squared_error(y_test, pred["Climatología"])  # referencia para la habilidad
filas = {n: metricas(y_test, pred[n], ESCALA, mse_clim) for n in BASES + NOMBRES}  # seis métodos
tabla_H = pd.DataFrame(filas).T
for n in NOMBRES:
    tabla_H.loc[n, "t_ajuste_s"] = HP[n]["t_ajuste_s"]  # tiempo del ajuste final (Etapa 4)
tabla_H.to_csv(TAB / "tabla_H_metricas_prueba.csv")     # para el reporte
print(f"Modelo elegido en validación: {ELEGIDO} · RMSE de prueba {tabla_H.loc[ELEGIDO, 'RMSE']:.2f} mm · "
      f"climatología {tabla_H.loc['Climatología', 'RMSE']:.2f} mm · n = {len(y_test)}")  # ← Anote (Tabla H: n, en la tabla pequeña)
tabla_H.round({"n": 0, "MSE": 1, "RMSE": 2, "MAE": 2, "R2": 3, "RSS": 0, "MedAE": 2, "MaxErr": 1, "VarExp": 3,
               "sMAPE_%": 1, "MASE": 3, "Hab_clim": 3, "t_ajuste_s": 2})  # ← Anote (Tabla H: las columnas de la tabla)

**Cómo leer la Tabla H.** Empiece por la fila del modelo elegido en validación y pregúntese tres cosas. (1) ¿Supera a
la **mejor** línea base, no a la peor? (2) ¿La diferencia es grande comparada con el propio RMSE, o es de décimas de
milímetro? (3) ¿El orden cambia si mira el MAE o la MedAE en vez del RMSE? Si cambia, la métrica elegida decide al
ganador, y eso debe declararse. La Etapa 6 añade una prueba estadística para la segunda pregunta. Si otro modelo
quedó mejor que el elegido, anótelo como observación: la regla de decisión no se cambia después de ver la prueba.

---
## 5.3 Diagnóstico de sobreajuste: dentro de muestra frente a prueba (Tabla I)

Un modelo que se ajusta muy bien a los datos con que aprendió y mal a los nuevos está **sobreajustado**. La comparación
directa es el error dentro de muestra (entrenamiento + validación, con el modelo final) contra el error de prueba. La
tabla indica cuántos meses hay en cada bloque.

In [ ]:
tabla_I = pd.DataFrame({n: {"n_ajuste": len(y_trval), "n_prueba": len(y_test),  # tamaños de los dos bloques
                            "RMSE_ajuste": root_mean_squared_error(y_trval, pred_trval[n]),  # error dentro de muestra
                            "RMSE_prueba": root_mean_squared_error(y_test, pred[n]),  # error en datos nuevos
                            "R2_ajuste": r2_score(y_trval, pred_trval[n]),  # frente a la media del desarrollo
                            "R2_prueba": r2_score(y_test, pred[n])}  # frente a la media de la prueba
                        for n in NOMBRES}).T
tabla_I["cociente_RMSE"] = tabla_I["RMSE_prueba"] / tabla_I["RMSE_ajuste"]  # > 1: peor en datos nuevos
tabla_I.to_csv(TAB / "tabla_I_sobreajuste.csv")
tabla_I.round(3)  # ← Anote (Tabla I: RMSE y R² de ajuste y prueba, y el cociente)

> **Ojo con la lectura ingenua.** El R² dentro de muestra se calcula contra la media del desarrollo (varias décadas) y
> el de prueba contra la media de la prueba (poco más de una década); las dos medias y varianzas son distintas, así que
> la diferencia de R² no mide sólo sobreajuste. Por eso la Tabla I incluye el RMSE, que está en milímetros en ambos
> casos, y el número de meses de cada bloque.

---
## 5.4 Serie real frente a pronóstico (Figura F8)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.4))
contexto = y_trval.iloc[-60:]                           # los últimos cinco años del desarrollo, como contexto
ax.plot(contexto.index, contexto, color="#B7C4BD", lw=0.8, label="real (ajuste)")
ax.plot(y_test.index, y_test, color=COLORES["real"], lw=1.4, label="real (prueba)")
for n in NOMBRES:
    ax.plot(pred.index, pred[n], lw=1.1, color=COLORES[n], label=n + (" (elegido en validación)" if n == ELEGIDO else ""))  # un color por modelo
ax.plot(pred.index, pred["Climatología"], lw=1.0, ls="--", color=COLORES["Climatología"], label="Climatología")
ax.axvline(y_test.index[0], color="k", ls=":", lw=0.8)  # inicio del bloque de prueba
ax.set(title="Precipitación real frente a pronóstico a un paso, bloque de prueba", xlabel="Fecha", ylabel="mm")
ax.legend(ncol=3, fontsize=8, loc="upper left")
guardar(fig, "F08_est2038_real_vs_pronostico")          # → Figura F8 del manual
plt.show()

---
## 5.5 Residuos y paridad (Figuras F9 y F10)

`PredictionErrorDisplay` de scikit-learn dibuja las dos vistas clásicas. En la de **residuos** (error contra valor
predicho) se busca una nube sin forma alrededor de cero; un embudo indica que el error crece con la lluvia
(heterocedasticidad) y una curva indica que el modelo no capturó parte de la relación. En la de **paridad**, el valor
**real va en el eje vertical** y el **predicho en el horizontal**; los puntos sobre la diagonal son aciertos. Los puntos
que quedan **por encima** de la diagonal son meses en que el valor real superó al predicho: **subestimación**. Si en
los meses grandes la nube se despega hacia arriba, el modelo subestima las tormentas, el comportamiento típico de un
regresor entrenado con error cuadrático sobre datos intermitentes.

In [ ]:
from sklearn.metrics import PredictionErrorDisplay
for tipo, archivo, titulo in [("residual_vs_predicted", "F09_est2038_residuos", "Residuos"),
                              ("actual_vs_predicted", "F10_est2038_paridad", "Paridad")]:
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
    for a, n in zip(ax, NOMBRES):
        PredictionErrorDisplay.from_predictions(y_test, pred[n], kind=tipo, ax=a,  # paridad: x = predicho, y = real
                                                scatter_kwargs={"alpha": 0.6, "color": COLORES[n]})
        a.set_title(f"{titulo} · {n}")
        a.set_xlabel("Valor predicho (mm)")  # rótulos en español (scikit-learn los pone en inglés)
        a.set_ylabel("Residuo: real − predicho (mm)" if tipo.startswith("residual") else "Valor real (mm)")
    fig.tight_layout()
    guardar(fig, archivo)                    # → Figuras F9 y F10 del manual
    plt.show()

---
## 5.6 Curvas de aprendizaje con validación cruzada temporal (Figura F11)

Una curva de aprendizaje muestra cómo cambian el error de ajuste y el de validación al crecer el número de datos de
entrenamiento. La lectura es **condicional**: si la brecha entre ambas curvas es grande y no se cierra al añadir datos,
hay indicios de sobreajuste; si las dos curvas quedan altas y juntas, el modelo o las características no alcanzan a
capturar la relación (subajuste); si la curva de validación sigue bajando en el último punto, más datos ayudarían. Son
indicios, no veredictos: el ruido entre pliegues (las barras) puede ser del tamaño de la brecha.

> **Nota técnica (por qué la curva se construye a mano).** `sklearn.model_selection.learning_curve` toma como tamaño
> máximo el del **primer** pliegue y entrena con las **primeras** *n* observaciones. Con `TimeSeriesSplit` el primer
> pliegue es el más pequeño, así que la curva se detendría en una sexta parte de los datos sin avisar. Aquí se usa
> `TimeSeriesSplit(n_splits=5)` sobre entrenamiento + validación y, para cada fracción, se entrena con los meses **más
> recientes** de cada pliegue; el eje horizontal muestra el número absoluto de meses.

> **Un diagnóstico condicionado.** Las curvas usan los hiperparámetros que la Etapa 4 eligió con el bloque de
> validación, que es **posterior** a los primeros pliegues de `TimeSeriesSplit`. Para esos pliegues, los
> hiperparámetros «conocen» datos que están en su futuro. Las curvas sirven para entender el comportamiento del modelo
> elegido, no para estimar de forma limpia su error; esa estimación limpia es la validación anidada de la Etapa 6.

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

def estimador(nombre):
    """Reconstruye el modelo de la Etapa 4 con sus hiperparámetros elegidos."""
    base = {"XGBoost": XGBRegressor(objective="reg:squarederror", tree_method="hist", random_state=SEMILLA, n_jobs=-1),
            "SVR": TransformedTargetRegressor(regressor=make_pipeline(StandardScaler(), SVR(kernel="rbf")),
                                              transformer=StandardScaler()),
            "Random Forest": RandomForestRegressor(random_state=SEMILLA, n_jobs=-1)}[nombre]
    return base.set_params(**HP[nombre]["params"])

print(f"Hiperparámetros elegidos con el bloque de validación {y_val.index[0]:%Y-%m} a {y_val.index[-1]:%Y-%m}")
tscv = TimeSeriesSplit(n_splits=5)                   # 5 pliegues; cada uno valida en el bloque siguiente a su entrenamiento
FRACCIONES = np.linspace(0.2, 1.0, 5)                # 20 %, 40 %, …, 100 % de la ventana de cada pliegue
curvas = {}
for n in NOMBRES:
    filas = []
    for f in FRACCIONES:
        e_aj, e_va, tam = [], [], []
        for itr, iva in tscv.split(X_trval):
            sel = itr[-max(int(f * len(itr)), 24):]  # los meses MÁS RECIENTES antes de validar
            m = estimador(n).fit(X_trval.iloc[sel], y_trval.iloc[sel])
            e_aj.append(root_mean_squared_error(y_trval.iloc[sel], m.predict(X_trval.iloc[sel])))  # error de ajuste
            e_va.append(root_mean_squared_error(y_trval.iloc[iva], m.predict(X_trval.iloc[iva])))  # error de validación
            tam.append(len(sel))
        filas.append({"meses_medios": np.mean(tam), "RMSE_ajuste": np.mean(e_aj), "RMSE_val": np.mean(e_va),
                      "DE_val": np.std(e_va)})
    curvas[n] = pd.DataFrame(filas)

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
for a, n in zip(ax, NOMBRES):
    d = curvas[n]
    a.plot(d["meses_medios"], d["RMSE_ajuste"], "o-", color=COLORES[n], label="ajuste")
    a.errorbar(d["meses_medios"], d["RMSE_val"], yerr=d["DE_val"], fmt="s--", color="k", capsize=3,
               label="validación (media ± DE de 5 pliegues)")
    a.set(title=f"Aprendizaje · {n}", xlabel="Meses de entrenamiento (media por pliegue)")
ax[0].set_ylabel("RMSE (mm)"); ax[0].legend(fontsize=8)
fig.tight_layout()
guardar(fig, "F11_est2038_curvas_aprendizaje")       # → Figura F11 del manual
plt.show()
print({n: curvas[n].round(2).iloc[-1].to_dict() for n in NOMBRES})  # último punto de cada curva

---
## 5.7 Curvas de validación de un hiperparámetro (Figura F12)

`validation_curve` sí funciona bien con `TimeSeriesSplit`: para cada valor de un hiperparámetro reentrena en los cinco
pliegues y promedia. Se barre la profundidad máxima en XGBoost y Random Forest, y `C` en la SVR. Lo esperable es que
la curva de ajuste mejore al crecer la complejidad; el punto de interés es dónde la de validación **deja de mejorar o
empeora**. Si la curva de validación es plana dentro de sus barras, el hiperparámetro importa poco en este rango y la
elección de la Etapa 4 no es crítica; dígalo así en su reporte.

In [ ]:
from sklearn.model_selection import validation_curve
BARRIDOS = {"XGBoost": ("max_depth", [1, 2, 3, 4, 6, 8]),
            "SVR": ("regressor__svr__C", list(np.logspace(-2, 2, 9))),
            "Random Forest": ("max_depth", [1, 2, 3, 5, 8, 12, 20])}
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
mejor_val = {}
for a, n in zip(ax, NOMBRES):
    param, rango = BARRIDOS[n]
    s_aj, s_va = validation_curve(estimador(n), X_trval, y_trval, param_name=param, param_range=rango,
                                  cv=tscv, scoring="neg_root_mean_squared_error", n_jobs=1)  # puntajes negativos
    a.plot(rango, -s_aj.mean(axis=1), "o-", color=COLORES[n], label="ajuste")
    a.errorbar(rango, -s_va.mean(axis=1), yerr=s_va.std(axis=1), fmt="s--", color="k", capsize=3, label="validación")
    if n == "SVR":
        a.set_xscale("log")                    # C se explora en escala logarítmica
    a.set(title=f"Validación · {n}", xlabel=param.split("__")[-1])
    mejor_val[n] = rango[int(np.argmin(-s_va.mean(axis=1)))]  # valor con menor RMSE medio de validación
ax[0].set_ylabel("RMSE (mm)"); ax[0].legend(fontsize=8)
fig.tight_layout()
guardar(fig, "F12_est2038_curvas_validacion")  # → Figura F12 del manual
plt.show()
print("Valor con menor RMSE medio de validación:", {k: float(v) for k, v in mejor_val.items()})  # ← Anote (Tabla I)

---
## 5.8 Gráfico comparativo (Figura F13)

In [ ]:
orden = tabla_H.sort_values("RMSE").index         # de menor a mayor RMSE
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(orden))
ax.bar(x - 0.2, tabla_H.loc[orden, "RMSE"], 0.4, color="#17795C", label="RMSE")
ax.bar(x + 0.2, tabla_H.loc[orden, "MAE"], 0.4, color="#D9A21B", label="MAE")
ax.set_xticks(x, [o + ("\n(elegido en validación)" if o == ELEGIDO else "") for o in orden], rotation=15, fontsize=8)
ax.set(title=f"Comparación en el bloque de prueba (menor es mejor; n = {len(y_test)} meses)", ylabel="mm")
ax.legend()
guardar(fig, "F13_est2038_comparacion_metricas")  # → Figura F13 del manual
plt.show()
registrar("etapa5", {"elegido_en_validacion": ELEGIDO, "n_prueba": int(len(y_test)),
                     "huecos_calendario_prueba": [f"{t:%Y-%m}" for t in huecos],
                     "escala_mase": round(ESCALA, 4), "escala_mase_pares": N_PARES,
                     "escala_mase_hasta": f"{y_val.index[-1]:%Y-%m}",
                     "tabla_H": tabla_H.round(4).to_dict(orient="index"),
                     "tabla_I": tabla_I.round(4).to_dict(orient="index"),
                     "ceros_prueba": ceros, "mape_rf": float(mape_rf),
                     "curva_aprendizaje_final": {n: curvas[n].round(4).iloc[-1].to_dict() for n in NOMBRES},
                     "mejor_valor_curva_validacion": {k: float(v) for k, v in mejor_val.items()}})

---
## Cierre de la Etapa 5

> **Punto de verificación.** Tiene completas las Tablas H (seis filas, con el n y la escala del MASE) e I (tres filas) y guardó las figuras F8 a F13. Antes de cerrar, escriba en una línea cómo le fue en la prueba al modelo elegido en validación, con el RMSE como métrica principal y el MAE como secundaria, y si supera a la climatología: esa línea abre la discusión de su reporte.

**Siguiente (extraclase):** `P3_E6_Extraclase_Validacion_Rigurosa.ipynb` (Anexo P del manual).

In [ ]:
minutos = (time.perf_counter() - T0) / 60  # tiempo de pared desde el Bloque 0
registrar("etapa5_duracion_min", round(minutos, 2))
print(f"Etapa 5 terminada en {minutos:.1f} min · resultados en", RES.relative_to(BASE))  # ← Anote (Tabla A): duración